# 🛡️ AAMAS 2027: Session 2 — Byzantine Robustness, Scalability & Edge Vision
### Kaggle Account 2 (13-Hour GPU Budget)

**Paper Target:** *Hierarchical Ensemble Personalization in Federated Learning (HEP-FL)* (AAMAS 2027)  
**Dataset:** **CIFAR-100** (Pre-attached in `/kaggle/input`, auto-linked in 0.0s)  
**Target Hardware:** Tesla T4 GPU (16 GB VRAM)  
**Assigned Jobs:**
1. **Job 2 (Table 2)**: Byzantine Multi-Attack Robustness Suite (4 Defenses × 4 Attacks × 5 Byzantine Rates = 80 runs) at **100 rounds**.
2. **Job 3 (Table 3)**: 50-Client Partial Participation Scaling ($C_p=0.2$, 10 active/round) with S-AFR.
3. **Job 4 (Table 4)**: MobileNetV3 Edge Vision Latency and Peak VRAM Profiling.

> 💡 **Tip:** Set your Kaggle notebook accelerator to **GPU T4 x1**, then click **Save Version -> "Save & Run All (Commit)"** to run in the background!


## 1. Environment Detection & Codebase Setup
Pulls the latest verified repository with fault-tolerant checkpointing and Kaggle dataset detection.


In [ ]:
import os, sys, shutil

ROOT = '/kaggle/working/Topology-aware-FDL'
OUT_DIR = '/kaggle/working/outputs'

if not os.path.exists(ROOT):
    print("🚀 Cloning repository from GitHub...")
    !git clone https://github.com/nam200718/Topology-aware-FDL.git {ROOT}
else:
    print("🔄 Repository exists, pulling latest updates from main...")
    !cd {ROOT} && git pull origin main

%cd {ROOT}
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print("✅ Current Working Directory:", os.getcwd())


## 2. Hardware Acceleration & Environment Verification
Verifies GPU acceleration and sets fast cuDNN primitives.


In [ ]:
!pip -q install pydantic pyyaml torchvision pandas matplotlib seaborn scipy
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM Capacity: {vram_gb:.2f} GB")
    torch.backends.cudnn.benchmark = True
else:
    print("⚠️ No GPU detected! Please navigate to Notebook Settings -> Accelerator -> GPU T4 x1.")


## 3. ⚡ Pre-loaded Kaggle Dataset Auto-Discovery (0.0s)
Directly verifies and symlinks `cifar-100-python` from `/kaggle/input` with zero network download.


In [ ]:
from src.data.dataset import _auto_link_dataset

data_dir = "./data"
found = _auto_link_dataset(data_dir, "cifar100")
if found:
    print("⚡ [Kaggle Input] CIFAR-100 dataset successfully verified and linked! (0.0s latency, 0 downloads)")
else:
    print("ℹ️ Note: If CIFAR-100 is attached under /kaggle/input, the experiment runner will automatically link it.")


## 4. 🛡️ Job 2: Byzantine Multi-Attack Robustness Benchmark (Table 2)
Evaluates 4 defenses (`fedavg`, `multikrum`, `ditto`, `topo_defended`) against 4 Byzantine attack types (`label_flip`, `sign_flip`, `gradient_ascent`, `random_noise`) across 5 corruption rates ($f \in [0.0, 0.1, 0.2, 0.3, 0.4]$) at **100 communication rounds**.


In [ ]:
# Run Job 2: Byzantine Robustness Suite (Table 2)
!python -m src.baselines.run_all_baselines \
    --track byzantine \
    --methods fedavg multikrum ditto topo_defended \
    --attacks label_flip sign_flip gradient_ascent random_noise \
    --rates 0.0 0.1 0.2 0.3 0.4 \
    --rounds 100 \
    --seeds 42


### Inspect Job 2 Results Table


In [ ]:
import pandas as pd

b_csv = "outputs/baselines/byzantine/results_byzantine.csv"
if os.path.exists(b_csv):
    df_b = pd.read_csv(b_csv)
    print(f"✅ Job 2 completed successfully! Total evaluated cells: {len(df_b)}")
    display(df_b[["method", "attack", "byzantine_rate", "mean_acc", "mean_b10", "elapsed_s"]])
else:
    print("⚠️ Checkpoint file not yet created.")


## 5. 👥 Job 3: 50-Client Scalability Benchmark with S-AFR (Table 3)
Evaluates 50 clients with partial participation ($C_p=0.2$, 10 active clients/round) under Moderate (0.5) and Severe (0.1) Dirichlet non-IID on CIFAR-100.


In [ ]:
# Run Job 3: 50-Client Scale Benchmark (Table 3)
!python scripts/run_scale_50clients.py \
    --dataset cifar100 \
    --rounds 50 \
    --clients 50 \
    --clients-per-round 10


## 6. 📱 Job 4: MobileNetV3 Edge Vision Latency & Peak VRAM Profiling (Table 4)
Profiles physical edge hardware metrics (latency, parameters, peak VRAM) and convergence under non-IID conditions on CIFAR-100.


In [ ]:
# Run Job 4: MobileNetV3 Edge Hardware Profiling (Table 4)
!python scripts/run_mobilenet_benchmark.py \
    --dataset cifar100 \
    --rounds 30 \
    --clients 15


## 7. 📦 Packaging Account 2 Artifacts
Compresses all CSV and JSON results from Account 2 into a single zip file for 1-click download.


In [ ]:
import zipfile

zip_target = "/kaggle/working/account2_artifacts.zip"
files_to_pack = [
    "outputs/baselines/byzantine/results_byzantine.csv",
    "outputs/baselines/byzantine/results_byzantine.json",
    "outputs/scale_50clients_results.json",
    "outputs/mobilenet_benchmark_results.json",
]

with zipfile.ZipFile(zip_target, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in files_to_pack:
        if os.path.exists(f):
            zf.write(f, arcname=os.path.basename(f))

print(f"🎉 All Account 2 artifacts packaged into: {zip_target}")
print(f"File Size: {os.path.getsize(zip_target) / 1024:.1f} KB")
print("👉 You can now download 'account2_artifacts.zip' directly from the Kaggle Output file explorer on the right!")
